In [4]:
import os
import sys

current_dir = os.getcwd()
# Make <repo>/src importable when running from the workspace (no wheel installed)
src_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "..", "src"))

if src_dir not in sys.path:
    sys.path.append(src_dir)

In [5]:
from dab_project.citibike.citibike_utils import get_trip_duration_mins
from dab_project.utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [6]:
# Defaults for local/interactive runs; job base_parameters override these.
dbutils.widgets.text("pipeline_id", "local")
dbutils.widgets.text("run_id", "local")
dbutils.widgets.text("task_id", "local")
dbutils.widgets.text("processed_timestamp", "local")
dbutils.widgets.text("catalog", "citibike_dev")

pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [7]:
df = spark.read.table(f"{catalog}.01_bronze.jc_citibike")

In [8]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [9]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [10]:
df = df.withColumn("metadata", 
              create_map(
                  lit("pipeline_id"), lit(pipeline_id),
                  lit("run_id"), lit(run_id),
                  lit("task_id"), lit(task_id),
                  lit("processed_timestamp"), lit(processed_timestamp)
                  ))

In [11]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
    )

In [12]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.02_silver.jc_citibike")